# 01 · Data audit

Everything downstream depends on the panel being what it claims to be: genuine
consolidated 5-minute bars, on real exchange sessions, in New York wall-clock
time. This notebook checks that before any return is computed.

Run `make data && make pipeline` first.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from closingbell import config as C, calendar_utils as cal

def table(name):
    return pd.read_csv(C.TABLES / f"{name}.csv")

sessions = pd.read_parquet(C.PROCESSED / "sessions.parquet")
print(f"{len(sessions):,} ticker-sessions, {sessions.session.min()} to {sessions.session.max()}")

15,756 ticker-sessions, 2021-01-04 to 2026-03-31


## The exchange calendar

Sessions come from XNYS, not from the file. Holidays are absent by construction and early closes expect 42 bars, not 78.

In [2]:
schedule = cal.session_schedule(str(sessions.session.min()), str(sessions.session.max()))
print("sessions:", len(schedule))
print("early closes:", int(schedule.is_early_close.sum()))
schedule[schedule.is_early_close][["close_time", "expected_bars"]]

sessions: 1316
early closes: 10


,close_time,expected_bars
session,,
2021-11-26,13:00:00,42
2022-11-25,13:00:00,42
2023-07-03,13:00:00,42
2023-11-24,13:00:00,42
2024-07-03,13:00:00,42
2024-11-29,13:00:00,42
2024-12-24,13:00:00,42
2025-07-03,13:00:00,42
2025-11-28,13:00:00,42


In [3]:
print("holidays inside the sample (weekdays the exchange was shut):")
h = cal.holidays(str(sessions.session.min()), str(sessions.session.max()))
print(len(h), "->", [str(d.date()) for d in h[:8]], "...")

holidays inside the sample (weekdays the exchange was shut):
51 -> ['2021-01-18', '2021-02-15', '2021-04-02', '2021-05-31', '2021-07-05', '2021-09-06', '2021-11-25', '2021-12-24'] ...


## Daylight saving

The clocks change on a Sunday, so the Monday session is still 390 regular-hours minutes. This table demonstrates that rather than assuming it.

In [4]:
table('dst_sessions').head(12)

,session,dst_transition,tickers,expected_bars,median_observed,min_observed
0,2021-03-15,spring_forward,12,78,78.0,78
1,2021-11-08,fall_back,12,78,78.0,76
2,2022-03-14,spring_forward,12,78,78.0,78
3,2022-11-07,fall_back,12,78,78.0,78
4,2023-03-13,spring_forward,12,78,24.5,22
5,2023-11-06,fall_back,12,78,78.0,78
6,2024-03-11,spring_forward,12,78,78.0,78
7,2024-11-04,fall_back,12,78,78.0,78
8,2025-03-10,spring_forward,12,78,78.0,78
9,2025-11-03,fall_back,12,78,78.0,78


## Early closes

An early close genuinely has no 15:30-16:00 window. It is excluded as an *event* and retained as a *following* session.

In [5]:
table('early_close_sessions')

,session,close_time,tickers,expected_bars,median_observed,bars_after_1300
0,2021-11-26,13:00:00,12,42,42.0,0
1,2022-11-25,13:00:00,12,42,42.0,0
2,2023-07-03,13:00:00,12,42,42.0,0
3,2023-11-24,13:00:00,12,42,42.0,0
4,2024-07-03,13:00:00,12,42,42.0,0
5,2024-11-29,13:00:00,12,42,42.0,0
6,2024-12-24,13:00:00,12,42,42.0,0
7,2025-07-03,13:00:00,12,42,42.0,0
8,2025-11-28,13:00:00,12,42,42.0,0
9,2025-12-24,13:00:00,12,42,42.0,0


## Completeness and exclusions

In [6]:
table('session_quality_summary').round(4)

,ticker,calendar_sessions,sessions_with_data,expected_bars,observed_bars,usable_sessions,event_eligible_sessions,auction_coverage,missing_bar_pct
0,AAPL,1316,1312,102288,101746,1303,1287,0.9932,0.0053
1,AMZN,1316,1312,102288,101710,1302,1286,0.9924,0.0057
2,GOOGL,1316,1297,102288,100376,1285,1274,0.9810,0.0187
3,IWM,1316,1313,102288,101741,1304,1292,0.9939,0.0053
4,JPM,1316,1315,102288,101899,1306,1294,0.9939,0.0038
5,META,1316,1314,102288,101749,1303,1292,0.9939,0.0053
6,MSFT,1316,1315,102288,101786,1303,1290,0.9947,0.0049
7,NVDA,1316,1315,102288,101783,1303,1290,0.9932,0.0049
8,QQQ,1316,1316,102288,101895,1307,1292,0.9954,0.0038
9,SPY,1316,1316,102288,101897,1305,1292,0.9954,0.0038


In [7]:
ex = table('session_exclusions')
print(ex.to_string(index=False))
print()
qc = table('session_quality')
print(f"usable: {int(qc.usable.sum()):,} of {len(qc):,} ticker-sessions "
      f"({qc.usable.mean():.2%})")
print(f"expected bars {qc.expected_bars.sum():,}, observed {qc.observed_bars.sum():,} "
      f"-> {1 - qc.observed_bars.sum()/qc.expected_bars.sum():.3%} missing")

  exclusion_reason  sessions  tickers
incomplete_session       127       12
           no_data        36        9
   erroneous_print         2        2
 missing_core_slot         1        1

usable: 15,626 of 15,792 ticker-sessions (98.95%)
expected bars 1,227,456, observed 1,220,267 -> 0.586% missing


## Cross-validation against an independent vendor

The 5-minute panel is aggregated back to daily bars and compared with daily data
from a separate source, after undoing the vendor's split adjustment. Prices that
agree to well under a basis point, and volume that reconciles to the fraction
extended-hours trading accounts for, is what real consolidated data looks like.

In [8]:
xv = table('data_crossvalidation')
xv[["ticker", "n_sessions", "open_median_bps", "close_median_bps", "close_p99_bps",
    "close_over_50bp", "volume_ratio_median"]].round(3)

,ticker,n_sessions,open_median_bps,close_median_bps,close_p99_bps,close_over_50bp,volume_ratio_median
0,AAPL,1312,0.725,0.652,17.248,2,0.941
1,AMZN,1312,0.660,0.000,17.994,4,0.943
2,GOOGL,1297,0.480,0.000,12.068,3,0.931
3,IWM,1313,0.000,0.000,10.290,4,0.935
4,JPM,1315,0.000,0.000,18.751,5,0.954
5,META,1314,0.880,0.000,11.691,3,0.950
6,MSFT,1315,1.154,0.000,16.331,4,0.936
7,NVDA,1315,0.662,0.001,24.776,4,0.949
8,QQQ,1316,0.000,0.822,10.108,3,0.916
9,SPY,1316,0.000,0.755,5.432,4,0.883


Volume ratios below one are expected: vendor daily volume includes pre- and post-market trading that the regular-hours panel excludes.

In [9]:
worst = table('data_crossvalidation_worst')
print("largest close disagreements:")
worst.head(10).round(3)

largest close disagreements:


,ticker,session,p_1600,d_close_raw,close_err_bps,close_is_auction,usable
0,META,2023-03-14,190.68,194.02,172.147,False,False
1,NVDA,2023-03-14,237.04,240.63,149.192,False,False
2,TSLA,2023-03-14,180.76,183.26,136.418,False,False
3,AMZN,2023-03-14,93.59,94.88,135.961,False,False
4,AMZN,2023-03-13,93.65,92.43,131.992,False,False
5,AMZN,2022-09-30,114.46,113.00,129.204,False,False
6,AAPL,2022-09-30,139.96,138.20,127.352,False,False
7,XOM,2023-03-10,109.08,107.78,120.616,False,False
8,MSFT,2023-03-13,256.84,253.92,114.997,False,False
9,AAPL,2023-03-14,150.86,152.59,113.375,False,False


## Corporate actions

The feed is unadjusted, so an uncorrected 10:1 split reads as a −90% overnight
return. Splits are removed and cash dividends added back.

In [10]:
from closingbell.external import load_external
ext = load_external()
print(ext["splits"].to_string(index=False))
print()
sp = sessions[sessions.n_split_ratio.fillna(1) != 1][["ticker","session","p_1600","n_open_0930",
                                                      "r_overnight_raw","r_overnight"]]
print("overnight returns across split ex-dates, raw vs corrected:")
sp.round(4).to_string(index=False)

ticker    session  split_ratio
  NVDA 2021-07-20          4.0
  NVDA 2024-06-10         10.0
  AMZN 2022-06-06         20.0
 GOOGL 2022-07-18         20.0
  TSLA 2022-08-25          3.0



overnight returns across split ex-dates, raw vs corrected:


'ticker    session  p_1600  n_open_0930  r_overnight_raw  r_overnight\n  AMZN 2022-06-03 2447.00      125.245          -0.9488       0.0237\n GOOGL 2022-07-15 2235.55      112.610          -0.9496       0.0074\n  NVDA 2021-07-19  751.19      187.270          -0.7507      -0.0028\n  NVDA 2024-06-07 1208.88      120.350          -0.9004      -0.0045\n  TSLA 2022-08-24  891.29      302.380          -0.6607       0.0178'

## Auction capture

The bar stamped at the official close is only accepted as the auction if it is large enough to plausibly be one.

In [11]:
print("auction accepted on %.1f%% of ticker-sessions" % (100 * sessions.close_is_auction.mean()))
print("auction bars rejected as implausible: %d" % int(sessions.auction_rejected.sum()))
sessions.groupby("ticker")[["close_is_auction", "auction_rejected"]].mean().round(3)

auction accepted on 93.2% of ticker-sessions
auction bars rejected as implausible: 571


,close_is_auction,auction_rejected
ticker,,
AAPL,0.995,0.001
AMZN,0.995,0.000
GOOGL,0.995,0.000
IWM,0.995,0.002
JPM,0.992,0.003
META,0.995,0.000
MSFT,0.995,0.001
NVDA,0.994,0.000
QQQ,0.981,0.014
